# Reading texts through theoretical lenses; reading relations through network analysis

Social science has two distinct analytical traditions. One is **interpretive**: researchers bring a theoretical perspective to read materials—Foucault asks "how does this discourse produce power and knowledge", Bourdieu asks "what position does the actor occupy in the field, and what capital determines this position", Weber uses "ideal types" (deliberately exaggerated as a measuring rod) to quantify how far a real case deviates from it. This tradition has no off-the-shelf software; a function like `foucault()` does not exist in any R/Python package—scholars rely on reading, note-taking, and manual coding to erect claims piece by piece. The other is **computational**: social network analysis turns "who relates to whom" into a graph, then uses quantified metrics—degree centrality, betweenness, eigenvector centrality, community detection—to compute "who matters, who are the brokers, who clusters together." This tradition has `networkx` / `igraph` / `Gephi`.

This tutorial runs both traditions through a single workflow: first, read a batch of interviews and cases through three theoretical lenses, then perform centrality and community analysis on a help-seeking network. The three lenses appear abstract, but each has an actionable scaffold when operationalized—Foucault is a fixed interrogative grid where each axis points back to specific text fragments; Bourdieu projects an actor × capital table into a positional space as a scatter plot; Weber assigns each case a score on each dimension and calculates its distance from the pure type. The network analysis step is straightforward graph computation. A methodological requirement runs through the entire process: **any claim must be locatable and contestable**—each Foucaldian axis carries `support_units` that triggered it; each Bourdieusian coordinate traces back to original capital values; each Weberian deviation carries the original score. This is precisely what the non-software tradition most easily loses, and most needs to preserve.

We complete the full pipeline using `socialverse`, an analytics library for social science that implements these theoretical lenses and network analyses as functions with evidence anchors. All data are synthesized and built-in: interview fragments for Foucault/Bourdieu lenses, an actor × capital table for Bourdieu's field, an organization × Weber-bureaucracy-three-dimensions case table for ideal types, and a seven-person help-seeking edge list. For methodological background, see Foucault's *The Archaeology of Knowledge*, Bourdieu's *Distinction*, Weber's *Economy and Society*, and Wasserman & Faust's *Social Network Analysis*.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import os

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from matplotlib import font_manager

import socialverse as sv
from socialverse import datasets as ds

# 让图里的中文标题正常渲染:挑一个装了的 CJK 字体(缺则回落,不报错)
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _cjk in ("Arial Unicode MS", "Songti SC", "STHeiti", "Hiragino Sans GB",
             "PingFang SC", "Noto Sans CJK SC", "Microsoft YaHei"):
    if _cjk in _installed:
        plt.rcParams["font.sans-serif"] = [_cjk, "DejaVu Sans"]
        break
plt.rcParams["axes.unicode_minus"] = False

# 图存到 .py 同目录(而非运行时 cwd),这样下方 markdown 的 ![](fig.png) 才指得对
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # notebook 内核里没有 __file__
    _HERE = os.getcwd()


def fig_path(name):
    return os.path.join(_HERE, name)

## Loading materials

Both Foucault and Bourdieu lenses start from text, so the first step is to prepare the corpus into a codable form. The built-in `load_corpus` provides three interview fragments; `build_corpus` cuts them into **codable units** with character offsets (each unit has an id like `int01:0-184`, recording which document and segment it comes from), and `code_themes` performs one pass of reflective thematic coding on these units. These two steps are identical to any other qualitative analysis tutorial—they are the generic starting point of text analysis.

Why do thematic coding first? Because the Foucault lens reads "material that has already been coded once", and Bourdieu's field later uses this batch of themes as a position space. Below we load the corpus into a research state, run the first two steps, and examine what units are produced.

In [2]:
st = sv.StudyState()
st.write("sources", "corpora", ds.load_corpus())  # 3 段访谈片段

sv.pp.build_corpus(st)   # 切成带字符偏移的可编码单元 → corpus['units', ...]
sv.tl.code_themes(st)    # 反身主题编码 → codes['themes', ...]

units = st.corpus["units"]
print("语料单元数:", len(units))
print("主题数:", len(st.codes["themes"]))

语料单元数: 3
主题数: 12


## Foucault: interrogative protocol for discourse

Foucault's discourse analysis is not statistics—it is a fixed **archaeological/genealogical** interrogative grid. `foucault_discourse` operationalizes this grid as six axes—discourse constitution, conditions of possibility, inclusion/exclusion, normalization, power-knowledge, subjectification. For each axis, it uses conservative keywords to locate matching segments in the corpus units, attaching them as `support_units` (evidence anchors) for that axis. The `claim` field is intentionally left blank: the specific interpretive proposition is for the researcher to fill in, but this claim **must** land on the named units. In this way, discourse analysis—traditionally completed through reading notes—becomes a traceable claim→evidence scaffold.

After execution, the output lives in `evidence['claim_evidence']`. First, let's see how many units it covers and what the overall stance is.

In [3]:
sv.tl.foucault_discourse(st)  # → evidence['claim_evidence']
ce_f = st.evidence["claim_evidence"]

print("透镜:", ce_f["lens"], "· 覆盖单元数:", ce_f["n_units"])
print("立场:", ce_f["stance"])

透镜: foucault_discourse · 覆盖单元数: 3
立场: 解释性阅读协议(archaeology/genealogy),非统计;每条主张须由 support_units 支撑


Arranging the six axes in a table is most intuitive: each row is one axis of Foucault analysis, `method` indicates whether it belongs to the archaeological or genealogical questioning, `n_support` is the number of units anchored to this axis, and `support_units` lists which specific text segments triggered it. The `question` column in the table is the Chinese phrasing of Foucault's question—when writing an interpretation, you are answering these questions, and the evidence for your answer must come from the units in the corresponding row.

In [4]:
df_f = pd.DataFrame([
    {
        "轴": r["axis"],
        "method": r["method"],
        "n_support": r["n_support"],
        "support_units": ", ".join(r["support_units"][:3]) + ("…" if len(r["support_units"]) > 3 else ""),
        "福柯之问": r["question"],
    }
    for r in ce_f["readings"]
])
df_f

,轴,method,n_support,support_units,福柯之问
0,discursive_formation,archaeology,3,"int01:0-184, int02:0-185, int03:0-177",哪些陈述被组织为一个话语对象?什么规则界定了可说的对象、概念与主体位置?
1,conditions_of_possibility,archaeology,3,"int01:0-184, int02:0-185, int03:0-177",是什么历史性的知识型(épistémè)使这一陈述得以成为『真』并被言说?
2,inclusion_exclusion,genealogy,3,"int01:0-184, int02:0-185, int03:0-177","该话语把什么纳入『正常/可说』,又把什么划为『异常/沉默』而排除?"
3,normalization,genealogy,3,"int01:0-184, int02:0-185, int03:0-177","通过何种规范、分类与测量,主体被度量、矫正并趋于常态?"
4,power_knowledge,genealogy,3,"int01:0-184, int02:0-185, int03:0-177",知识如何生产权力、权力又如何生产知识(pouvoir-savoir)?谁因此获得说真话的资格?
5,subjectivation,genealogy,3,"int01:0-184, int02:0-185, int03:0-177",话语把言说者/被言说者构成为何种主体(病人/罪犯/学生…)?自我如何被规训?


Looking at the complete entry for just the **power_knowledge** axis: it is a genealogical question, `claim` awaits the researcher's input, and three interview units are named as evidence anchors. When you later write "this discourse achieves power-knowledge through some form of expert credibility", the system can trace back exactly which text segments support this reading—the interpretation is therefore locatable and open to peer contestation.

In [5]:
pk = next(r for r in ce_f["readings"] if r["axis"] == "power_knowledge")
print("轴   :", pk["axis"])
print("method:", pk["method"])
print("问   :", pk["question"])
print("claim:", pk["claim"])
print("证据锚点 support_units:", pk["support_units"])

轴   : power_knowledge
method: genealogy
问   : 知识如何生产权力、权力又如何生产知识(pouvoir-savoir)?谁因此获得说真话的资格?
claim: 待填:关于『power_knowledge』的解读性主张(以下 unit 为其证据锚点)
证据锚点 support_units: ['int01:0-184', 'int02:0-185', 'int03:0-177']


## Bourdieu: positioning actors in the field

Bourdieu's **field** is a position space: actors are distributed within it according to their **capital** (economic, cultural, social, symbolic)—both total volume and structure. This step has real numerical values—`bourdieu_field` projects an actor × capital table into two dimensions (if `prince` is installed, it uses multiple correspondence analysis; otherwise it falls back to centered SVD-PCA implemented from scratch), assigning each actor a coordinate. The key concept is **homology**: each coordinate is traced back to that actor's own capital indicators, so the position is not arbitrary but interpretable as "because economic capital is high and cultural capital is low, it falls here."

We construct a small table: four actors' quantities on two capital dimensions—"economic capital / cultural capital". The industrialist has high economic and low cultural capital, the scholar the reverse, plus a centrist and a culture-leaning young artist. This table is the input to field analysis.

In [6]:
capital = pd.DataFrame(
    {"economic": [9, 2, 5, 3],
     "cultural": [2, 9, 5, 7]},
    index=["实业家", "学者", "中间派", "青年艺术家"],
)
capital

,economic,cultural
实业家,9,2
学者,2,9
中间派,5,5
青年艺术家,3,7


The Bourdieu lens, beyond reading the capital table above, also requires two things: theme coding (`code_themes` already filled it in, serving as the position space) and a name for each capital dimension. We declare these dimension names into the research state, then call the lens. The output lives in `models['field_map']`, containing the projection method, the variance explained by each of the two principal axes, and the two-dimensional coordinates for each actor.

In [7]:
st.write("variables", "constructs", ["economic", "cultural"])  # 资本维度的名字
sv.tl.bourdieu_field(st, capital_table=capital)                # 读主题 + 本表 → 位置空间
fm = st.models["field_map"]

print("投影方法:", fm["method"])
print("两主轴解释方差:", [round(v, 3) for v in fm["explained_variance"]])
print("axis 语义:", fm["axes"])

投影方法: PCA(centered-SVD fallback)
两主轴解释方差: [0.991, 0.009]
axis 语义: {'x': 'capital_axis_1 (总资本量/结构主轴)', 'y': 'capital_axis_2 (资本构成:经济↔文化)'}


The positional coordinates for each actor. The first principal axis explains the vast majority of variance (about 99%), corresponding to "total capital volume / structural axis"; the second corresponds to "capital composition: economic ↔ cultural". The industrialist falls at one end of the axis, the scholar at the other—precisely the classic picture from *Distinction* of opposition within the ruling class along lines of capital composition.

In [8]:
pd.DataFrame(fm["positions"], index=["axis_1", "axis_2"]).T.round(3)

,axis_1,axis_2
实业家,-5.662,-0.250
学者,4.235,-0.431
中间派,-0.701,0.366
青年艺术家,2.127,0.315


The homology scaffold lives in `evidence['claim_evidence']['homology']`, one entry per actor, with `capital_indicators` that trace the coordinate back to original capital values and a `claim` awaiting the researcher's fill. Look at the first entry, the "industrialist": its coordinate (−5.66, −0.25) is explicitly linked to economic=9, cultural=2, these two original indicators. `themes_available=True` means the position space is present, allowing you to further read the homology between "capital position ↔ thematic stance".

In [9]:
ce_b = st.evidence["claim_evidence"]
print("themes_available(立场空间是否在场):", ce_b["themes_available"])

h0 = ce_b["homology"][0]
print("\n行动者   :", h0["actor"])
print("位置坐标 :", [round(x, 3) for x in h0["position"]])
print("资本指标 :", h0["capital_indicators"])
print("待填 claim:", h0["claim"])

themes_available(立场空间是否在场): True

行动者   : 实业家
位置坐标 : [-5.662, -0.25]
资本指标 : {'economic': 9.0, 'cultural': 2.0}
待填 claim: 待填:实业家 的立场(position-taking)与其资本结构的同源性主张


Bourdieu's field is naturally meant to be drawn. We visualize `field_map['positions']` directly as a scatter plot, the horizontal axis as total capital volume / structural axis, vertical axis as capital composition (economic ↔ cultural), with each actor labeled. This is not a new call—just visualization of the coordinates produced above.

In [10]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))
pos = fm["positions"]
xs = [pos[a][0] for a in pos]
ys = [pos[a][1] for a in pos]
ax.scatter(xs, ys, s=220, c="#4C72B0", edgecolors="white", linewidths=1.5, zorder=3)
for a in pos:
    ax.annotate(a, (pos[a][0], pos[a][1]), textcoords="offset points",
                xytext=(8, 8), fontsize=11, fontweight="bold")
ax.axhline(0, color="#bbbbbb", lw=0.8, zorder=1)
ax.axvline(0, color="#bbbbbb", lw=0.8, zorder=1)
ev = fm["explained_variance"]
ax.set_xlabel(f"axis 1 · 总资本量/结构主轴  (解释方差 {ev[0]:.0%})")
ax.set_ylabel(f"axis 2 · 资本构成 经济↔文化  (解释方差 {ev[1]:.0%})")
ax.set_title("布迪厄场域:行动者的位置空间")
fig.tight_layout()
fig.savefig(fig_path("fig_bourdieu_field.png"), dpi=130, bbox_inches="tight")
plt.close(fig)
print("已保存 fig_bourdieu_field.png")

已保存 fig_bourdieu_field.png


![Bourdieu field scatter plot](fig_bourdieu_field.png)

The "industrialist" (high economic, low cultural) and the "scholar" (high cultural, low economic) fall at opposite ends of the axis, the "centrist" occupies the middle, and the "young artist" leans toward the cultural end. Each point's position can be traced through `capital_indicators`; it is not placed arbitrarily.

## Weber: ideal type as measuring rod

Weber's **ideal type** is an analytical construct deliberately exaggerated in one direction—no real case will fully match it, and **the degree to which a case deviates from the pure type** carries the interpretive weight. This is Verstehen (interpretive understanding): meaning emerges precisely in the deviation. `weber_ideal_type` makes this logic numerically real: you define a measuring rod by supplying a `schema` (each dimension gets a sentence describing "from opposite extreme to pure extreme"), then a case × dimension scoring table, and the function computes a proximity to the pure type for each case on each dimension (∈[0,1], 1=close to pure), with deviation = 1 − proximity.

There is one easy pitfall worth demonstrating first. Ideal types only make sense when **the dimensions in the schema are real numeric columns that exist in the case table**; if they don't align, the function will not error, but will compute NaN everywhere, coverage=0, with no interpretable deviations. Below we intentionally feed mismatched input to see how the diagnostics honestly tell you "it ran, but it's empty."

In [11]:
stw0 = sv.StudyState()
stw0.write("sources", "datasets", ds.load_survey())
sv.tl.weber_ideal_type(
    stw0,
    schema={"individualism": "collectivist..individualist", "hierarchy": "flat..hierarchical"},
    cases=ds.load_survey().head(6),   # 案例=survey 的 item 列,不含 schema 里的维度
)
print("声明的维度:", stw0.models["ideal_type"]["dimensions"])
print("coverage.overall(维度覆盖率):", stw0.diagnostics["coverage"]["overall"])
print("可解释的偏离条目数:", len(stw0.evidence["claim_evidence"]["deviations"]),
      "  ← 覆盖率 0,跑通但无意义")

声明的维度: ['individualism', 'hierarchy']
coverage.overall(维度覆盖率): 0.0
可解释的偏离条目数: 0   ← 覆盖率 0,跑通但无意义


Coverage is 0, the deviation list is empty—the function did not error, but `diagnostics['coverage']` quantified "whether it means anything" as a number. Now let's get it right: along Weber's **legal-rational authority (bureaucracy)** three axes—rule_bound, impersonal, hierarchy—score five organizations, with the pure extreme being "perfect bureaucracy". These three dimensions are the actual numeric columns in the case table, so coverage will be full and each case's deviation will compute correctly.

In [12]:
cases = pd.DataFrame(
    {"rule_bound": [9, 7, 2, 5, 8],   # 依成文规程的程度
     "impersonal": [8, 6, 1, 4, 9],   # 对事不对人(非人格化)
     "hierarchy":  [9, 5, 2, 6, 7]},  # 科层等级严格度
    index=["普鲁士文官系统", "现代企业", "先知运动", "创业公司", "天主教会"],
)
cases

,rule_bound,impersonal,hierarchy
普鲁士文官系统,9,8,9
现代企业,7,6,5
先知运动,2,1,2
创业公司,5,4,6
天主教会,8,9,7


Assign each dimension a sentence describing "opposite extreme .. pure extreme", with the pure extreme being perfect bureaucracy, then call the lens. The output `models['ideal_type']` contains per-case-per-dimension proximity scores and the average deviation for each case.

In [13]:
schema = {
    "rule_bound": "凭好恶裁量 .. 严格依成文规则(法理型纯粹极)",
    "impersonal": "对人不对事 .. 完全非人格化(对事不对人)",
    "hierarchy":  "扁平 .. 严格科层等级",
}
stw = sv.StudyState()
stw.write("sources", "datasets", cases)
sv.tl.weber_ideal_type(stw, schema=schema, cases=cases)
it = stw.models["ideal_type"]

print("维度:", it["dimensions"], "· 案例数:", it["n_cases"])
print("coverage.overall:", stw.diagnostics["coverage"]["overall"], "(=1.0,全维度可打分)")

维度: ['rule_bound', 'impersonal', 'hierarchy'] · 案例数: 5
coverage.overall: 1.0 (=1.0,全维度可打分)


Per-case-per-dimension proximity scores, where 1 means close to the "perfect bureaucracy" pure extreme. The Prussian civil service is close to 1 on all three dimensions, the most archetypal bureaucracy; the prophetic movement is low on all dimensions.

In [14]:
it["scores"]

,rule_bound,impersonal,hierarchy
普鲁士文官系统,1.0000,0.875,1.0000
现代企业,0.7143,0.625,0.4286
先知运动,0.0000,0.000,0.0000
创业公司,0.4286,0.375,0.5714
天主教会,0.8571,1.000,0.7143


Sort each case's average deviation from the pure type from largest to smallest: larger deviation means "more non-bureaucratic". The prophetic movement has the largest deviation, modern enterprises and startups are in the middle, and the Prussian civil service is closest to the pure type.

In [15]:
pd.Series(it["case_mean_deviation"], name="mean_deviation").round(3).sort_values(ascending=False)

先知运动       1.000
创业公司       0.542
现代企业       0.411
天主教会       0.143
普鲁士文官系统    0.042
Name: mean_deviation, dtype: float64

The three largest deviations are sorted first in `evidence['claim_evidence']['deviations']`, each entry recording "a certain case deviates most from the pure type on a certain dimension", with the original score `value` and a `claim` awaiting the researcher. The largest deviation is the **prophetic movement**—it approaches the opposite extreme on all three dimensions: rule_bound, impersonal, hierarchy—precisely what Weber used to contrast with legal-rational authority: **charismatic authority**. The meaning of ideal type emerges exactly in these deviations.

In [16]:
ce_w = stw.evidence["claim_evidence"]
for d in ce_w["deviations"][:3]:
    print(f"{d['case']} · {d['dimension']}  偏离={d['deviation']}  原始值={d['value']}")

先知运动 · rule_bound  偏离=1.0  原始值=2
创业公司 · impersonal  偏离=0.625  原始值=4
现代企业 · hierarchy  偏离=0.5714  原始值=5


Weber insists on **value neutrality (Wertfreiheit)**: the ideal type's scores are analytical "distance to the pure type", not an evaluation of the case's worth or merit. This lens encodes this commitment in the `governance['ethics']` slot—not as a comment, but as a structured record preserved alongside the analysis.

In [17]:
eth = stw.governance["ethics"]
print("原则:", eth["principle"])
print("声明:", eth["statement"])

原则: Wertfreiheit(价值中立)
声明: 本理想类型评分为分析性的『与纯粹类型的距离』,非对案例价值/优劣的评判;理想型是一侧强调的分析建构,任何真实案例都不应被期待与之完全吻合。


## Network: three centralities tell three stories

The first three steps apply interpretive lenses; the final step switches to real graph computation. `build_network` converts an **edge list** into a `networkx` graph and reads its structure: three centralities (degree, betweenness, eigenvector), density, and greedy-modularity community detection. The edge list must be supplied by the researcher; the function will not fabricate a network.

We construct a help-seeking/consultation network: seven researchers, who consults whom (weight = consultation frequency). It is deliberately designed with two loose clusters plus a bridge structure, so that the three centralities tell different stories—degree centrality measures "how many direct connections", betweenness centrality measures "whether you're a broker between others", eigenvector centrality measures "are you connected to important people".

In [18]:
edges = pd.DataFrame({
    "source": ["Ada", "Ben", "Cai", "Ada", "Cai", "Dev", "Eve", "Fei", "Dev", "Fei", "Cai", "Gao", "Eve", "Fei"],
    "target": ["Ben", "Cai", "Ada", "Cai", "Dev", "Eve", "Fei", "Dev", "Fei", "Eve", "Dev", "Fei", "Gao", "Gao"],
    "weight": [3, 2, 1, 2, 4, 1, 3, 2, 1, 2, 1, 1, 2, 1],
})
edges

,source,target,weight
0,Ada,Ben,3
1,Ben,Cai,2
2,Cai,Ada,1
3,Ada,Cai,2
4,Cai,Dev,4
5,Dev,Eve,1
6,Eve,Fei,3
7,Fei,Dev,2
8,Dev,Fei,1
9,Fei,Eve,2


Load the edge list into a new research state and build an undirected weighted graph. The output `models['network']` reports network size, density, average degree; `diagnostics['coverage']` records the connected component structure—here all seven nodes fall in a single connected component.

In [19]:
stn = sv.StudyState()
stn.write("sources", "datasets", edges)
sv.tl.build_network(stn, edges=edges, source="source", target="target",
                    weight="weight", directed=False, top_k=8)
net = stn.models["network"]

print("节点数:", net["n_nodes"], "· 边数:", net["n_edges"],
      "· 密度:", round(net["density"], 3), "· 平均度:", round(net["avg_degree"], 3))
cov = stn.diagnostics["coverage"]
print("连通分量:", cov["n_components"], "个 · 最大分量含", cov["largest_cc_size"], "个节点")

节点数: 7 · 边数: 9 · 密度: 0.429 · 平均度: 2.571
连通分量: 1 个 · 最大分量含 7 个节点


Arrange the three centralities side-by-side in a table—this is the teaching core of this step: on the same graph, the answer to "who is most important" varies depending on whether you ask about connection count, bridging power, or core embedding.

In [20]:
cent = net["centrality"]
nodes = sorted(set(cent["degree"]) | set(cent["betweenness"]) | set(cent["eigenvector"]))
df_c = pd.DataFrame({
    "degree(度)": pd.Series(cent["degree"]),
    "betweenness(介数)": pd.Series(cent["betweenness"]),
    "eigenvector(特征向量)": pd.Series(cent["eigenvector"]),
}).reindex(nodes).round(3)
df_c

,degree(度),betweenness(介数),eigenvector(特征向量)
Ada,0.333,0.000,0.588
Ben,0.333,0.000,0.588
Cai,0.500,0.533,0.522
Dev,0.500,0.633,0.142
Eve,0.500,0.000,0.082
Fei,0.500,0.267,0.074
Gao,0.333,0.000,0.050


The three centralities point to different people. **Degree centrality**: Cai / Dev / Eve / Fei are tied highest—the most direct connections. **Betweenness centrality**: Dev (0.63) and Cai (0.53) lead by far—they sit between the two clusters, acting as brokers; consultations must pass through them. This is exactly Burt's **structural hole** position. **Eigenvector centrality**: Ada / Ben (≈0.59) are highest instead—they are embedded in the tight Ada-Ben-Cai triangle; being "connected to important people counts as being important," not just having the most connections.

Community detection divides these seven into structural clusters. Greedy-modularity yields two communities with modularity Q≈0.43, matching the two clusters we designed the graph with.

In [21]:
comm = net["communities"]
print("社群数:", comm["n_communities"], "· 模块度 Q:", round(comm["modularity"], 3),
      "· 各社群规模:", comm["sizes"])

社群数: 2 · 模块度 Q: 0.433 · 各社群规模: [4, 3]


Finally, we visualize the network so the conclusions above are visible at a glance: node size by betweenness centrality (brokers larger), color by greedy-modularity community, edge width by consultation frequency. Coordinates come from `networkx`'s spring layout (fixed random seed for reproducibility), and all centrality and community values come from the `models['network']` output above; here we only rebuild the graph from the same edge list for layout and coloring.

In [22]:
from networkx.algorithms.community import greedy_modularity_communities

G = nx.from_pandas_edgelist(edges, "source", "target", edge_attr="weight")
btw = cent["betweenness"]

palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
parts = list(greedy_modularity_communities(G, weight="weight"))
node_color = {n: palette[ci % len(palette)] for ci, part in enumerate(parts) for n in part}

pos = nx.spring_layout(G, seed=1, weight="weight")
sizes = [300 + 4200 * btw.get(n, 0.0) for n in G.nodes()]   # 介数越高节点越大
colors = [node_color.get(n, "#999999") for n in G.nodes()]
widths = [0.6 + 0.9 * G[u][v]["weight"] for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(7.2, 5.6))
nx.draw_networkx_edges(G, pos, width=widths, edge_color="#c9c9c9", ax=ax)
nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=colors,
                       edgecolors="white", linewidths=1.5, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=11, font_weight="bold", ax=ax)
q = net["communities"].get("modularity")
ax.set_title(f"求助网络:节点大小=介数中心性,颜色=社群(Q≈{q:.2f})")
ax.axis("off")
fig.tight_layout()
fig.savefig(fig_path("fig_advice_network.png"), dpi=130, bbox_inches="tight")
plt.close(fig)
print("已保存 fig_advice_network.png")

已保存 fig_advice_network.png


![Help-seeking network](fig_advice_network.png)

Dev and Cai are visibly larger (high betweenness, sitting at the seam between the two colored communities—they are brokers), and the two colors correspond to the two communities detected by greedy-modularity. What the eye sees matches the numbers in the table exactly.

## Reproducible evidence chains

Working through this tutorial, there is one subtle difference from ordinary analysis scripts: each step, upon success, automatically records an entry in the research state, documenting which function it used, what it consumed, and what it produced. Foucault and Bourdieu share the same `st` (so it accumulates build_corpus → code_themes → foucault → bourdieu across four steps), while Weber and network use independent states. `st.summary()` tallies "which slots are filled, how many steps taken"—in social science, "what step and what data did this conclusion come from" often matters as much as the conclusion itself, and this evidence chain is kept for exactly this reason.

In [23]:
print("=== 主链 st(福柯 + 布迪厄共享)===")
print(st.summary())

=== 主链 st(福柯 + 布迪厄共享)===
StudyState
  sources: corpora
  variables: constructs
  corpus: documents, units, manifest
  codes: codebook, segments, themes, theme_map
  models: field_map
  evidence: provenance, claim_evidence
  artifacts: tables
  provenance: 4 step(s)


Expanding this ledger step by step, you can see how the requires → produces dependency of the entire chain is wired: `build_corpus` constructs codable units from raw corpus, `code_themes` consumes units and yields themes, the Foucault lens consumes units and yields evidence, the Bourdieu lens consumes themes and capital dimensions and yields the field.

In [24]:
for rec in st.provenance:
    pro = "+".join(f"{k}:{','.join(v)}" for k, v in rec["produces"].items()) or "∅"
    print(f"step {rec['step']}: {rec['function'].split('.')[-1]}  →  produces[{pro}]")

step 1: build_corpus  →  produces[corpus:documents,units,manifest+evidence:provenance]
step 2: code_themes  →  produces[codes:codebook,segments,themes,theme_map+evidence:claim_evidence+artifacts:tables]
step 3: foucault_discourse  →  produces[evidence:claim_evidence]
step 4: bourdieu_field  →  produces[models:field_map+evidence:claim_evidence]


## Summary

We have traversed two social science traditions through a single workflow: three theoretical lenses (Foucault's discourse interrogation, Bourdieu's field projection, Weber's ideal type scoring) read texts and cases, then perform centrality and community analysis on a help-seeking network. The network step is the counterpart to `networkx` / `igraph` / `Gephi`; the three lenses, by contrast, are counterparts to **scholarship traditions without software**—historically completed through reading, note-taking, manual coding.

Compared to pure network tools, this adds one thing throughout: it brings interpretation into the evidence chain. Each Foucaldian axis carries `support_units`, each Bourdieusian position traces to `capital_indicators`, each Weberian deviation carries the original score, and the coverage=0 example reminds you that "runs successfully" does not mean "makes sense"—interpretation is thus locatable and refutable, which is what the non-software tradition most easily loses and most needs to preserve. The next tutorial [08_governance_gates](08_governance_gates.ipynb) shifts to research governance: how ethics, bias, and compliance become real gatekeeping mechanisms.